# 06 · How much of the outcome is the inspector?

Notebooks 03–05 plateau at about twice the base rate, and a property's last score is a
weak guide to its next one. This notebook asks why, and tests four things:

1. **Inspector effects.** How much of a score depends on who inspected, measured out of
   sample.
2. **An inspector-adjusted target.** If inspector and year effects are removed from each
   score, do the same features predict the adjusted bottom 10% better?
3. **Inspector leniency in the temporal design.** A process feature: the assigned
   inspector's bottom-10 rate in earlier years.
4. **Regression instead of classification.** Fit the continuous score and rank by it.

Inspector identity is reconciled across Salesforce and Code Compliance's July 2026
backfill in `inspectors.py` (notebook 07) and appears only as codes in outputs.

## Findings

Run 2026-10-05 with reconciled inspector names. Outputs are cleared before commit; the
numbers below are from that run.

> **Corrected 2026-10-05.** The 2026-10-01 run treated typed variants of one inspector's name
> (for example a name with and without a suffix) as different inspectors. Inspector identity
> now comes from `inspectors.py` (see notebook 07). The main change: inspector leniency adds
> less to the temporal model than first reported (pooled PR-AUC 0.248, not 0.263). Notebooks
> 08–11 take this diagnosis further.

**A single inspection score barely persists, and the inspector moves it about as much as
the property's own record does.**

- **Spread across inspectors.** 50 inspectors scored an inspection; 36 have 30 or more
  (98% of visits). Bottom-10 rates run 0.5% at the 10th percentile, 7.5% at the median and
  27.7% at the 90th. Mean scores run 86.4 to 98.4 over the same range. Inspectors work across
  the city: 25–54% of an inspector's visits fall in the high-poverty third of tracts, against
  about a third of properties.
- **Out of sample.** I held out the latest inspection for 2,145 properties inspected in two
  or more fiscal years and fit a linear model to its score:
  - the property's earlier mean score explains 1.8% of the variance;
  - the held-out visit's inspector and year effect, estimated without the held-out visits,
    explains 2.9%;
  - both together explain 4.4%.
- **Bottom-10 status repeats mainly when the inspector does.** Across consecutive
  inspections in different fiscal years:
  - **same inspector:** a bottom-10 property is bottom-10 again 39% of the time, against 10%
    otherwise (524 pairs);
  - **different inspector:** 18% against 13% (2,685 pairs).
- **Removing inspector effects from the target does not make it more predictable.** The
  adjusted target agrees with the raw one for 93% of properties. The pre-window model's lift
  over random is 2.0× raw against 2.0× adjusted on the mirror, and 1.8× against 2.0× on
  grouped CV. Notebook 11 shows why: inspector effects carry some real condition signal.
- **Inspector leniency as a process feature.** In the temporal design, the assigned
  inspector's bottom-10 rate in earlier years alone scores pooled PR-AUC 0.158, close to the
  property's last score (0.171). Adding it to the best temporal model raises pooled PR-AUC
  from 0.230 to 0.248, a lift of 2.2× over the 0.110 base rate. It is usable only where
  assignments are known before the visit. Code assigns inspectors by area, adjusted for
  staffing and enforcement needs. It describes the visit and says nothing about the building.
- **Regression is worse than classification.** Fitting the continuous score (minimum score
  for properties, unit score for the temporal design) and ranking by prediction loses 2–5
  points of PR-AUC across the designs.

**What this means for "usable."** With this outcome, property-level models reach about 2×
the base rate, whatever the features. For comparison, Chelsea's published precision-recall
curve (read off its figure) sits near 0.55–0.6 precision against a 0.45 base rate, about
1.3×.

In [ ]:
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.linalg import lsqr

import features as f
import holdout as h
import temporal as t

warnings.filterwarnings("ignore", message="Skipping features without any observed values")
pd.set_option("display.float_format", "{:.3f}".format)

insp = t.load_inspections()
known = insp.inspector >= 0
print(f"scored inspections: {len(insp):,}; inspector known: {known.mean():.1%}; inspectors: {insp.loc[known, 'inspector'].nunique()}")
vc = insp.loc[known, "inspector"].value_counts()
print("inspections per inspector: median", int(vc.median()), "| inspectors with 30+:", int((vc >= 30).sum()),
      f"covering {insp.inspector.map(vc).ge(30).mean():.1%} of inspections")

## 1. Inspector effects

In [ ]:
# Bottom-10 rate by inspector (inspectors with 30+ inspections)
d = insp[insp.inspector.map(vc).fillna(0) >= 30]
by = d.groupby("inspector").agg(n=("bottom10", "size"), b10=("bottom10", "mean"), mean_score=("score", "mean"))
print("bottom-10 rate across inspectors, percentiles 10/25/50/75/90:",
      by.b10.quantile([.1, .25, .5, .75, .9]).round(3).tolist())
print("mean score across inspectors, percentiles 10/50/90:", by.mean_score.quantile([.1, .5, .9]).round(1).tolist())

# Do inspectors work everywhere? Share of each inspector's visits in high-poverty tracts.
base = h.load_base()
hi = (base.set_index("address").pct_poverty >= h.poverty_cut(base))
share_hi = d.assign(hi=d.address.map(hi)).groupby("inspector").hi.mean()
print("share of an inspector's visits in the high-poverty third, percentiles 10/50/90:",
      share_hi.quantile([.1, .5, .9]).round(2).tolist())

In [ ]:
def onehot(codes) -> sp.csr_matrix:
    c = pd.factorize(codes)[0]
    return sp.csr_matrix((np.ones(len(c)), (np.arange(len(c)), c)), shape=(len(c), c.max() + 1))


def two_way_effects(rows: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    # score = property + inspector + fiscal year + error; returns inspector and year effects
    # (centred) from a sparse least-squares fit. Unknown inspector gets its own level.
    y = rows.score.to_numpy(float) - rows.score.mean()
    P, I, F = onehot(rows.address), onehot(rows.inspector), onehot(rows.fiscal_year)
    beta = lsqr(sp.hstack([P, I, F]).tocsr(), y, atol=1e-10, btol=1e-10, iter_lim=50000)[0]
    nP, nI = P.shape[1], I.shape[1]
    ins = pd.Series(beta[nP:nP + nI], index=pd.factorize(rows.inspector)[1])
    fy = pd.Series(beta[nP + nI:], index=pd.factorize(rows.fiscal_year)[1])
    return ins - ins.mean(), fy - fy.mean()


# Out of sample: for properties inspected in 2+ fiscal years, hold out the latest
# inspection and predict its score from the property's earlier scores, with and without
# the held-out visit's inspector and year effects (estimated without the held-out visits).
ins_sorted = insp.sort_values(["address", "date_opened", "sr_number"])
multi_fy = ins_sorted.groupby("address").fiscal_year.transform("nunique") >= 2
last = ins_sorted[multi_fy].groupby("address").tail(1)
rest = ins_sorted.drop(last.index)
eff_i, eff_fy = two_way_effects(rest)
prior_mean = rest[rest.address.isin(last.address)].groupby("address").score.mean()
adj_rest = rest.score - rest.inspector.map(eff_i).fillna(0) - rest.fiscal_year.map(eff_fy).fillna(0)
prior_adj_mean = adj_rest[rest.address.isin(last.address)].groupby(rest.address).mean()

y = last.score.to_numpy(float)
a = last.address.map(prior_mean).to_numpy()
a_adj = last.address.map(prior_adj_mean).to_numpy()
ie = last.inspector.map(eff_i).fillna(0).to_numpy()
fe = last.fiscal_year.map(eff_fy).fillna(0).to_numpy()


def ols_r2(*cols):
    # share of held-out score variance explained by a linear fit on the given predictors
    # (an intercept plus one slope each, so no level shift or scale mismatch counts against them)
    X = np.column_stack([np.ones(len(y)), *cols])
    ok = np.isfinite(X).all(axis=1)
    beta, *_ = np.linalg.lstsq(X[ok], y[ok], rcond=None)
    r = y[ok] - X[ok] @ beta
    return 1 - (r @ r) / ((y[ok] - y[ok].mean()) ** 2).sum()


print(f"held-out latest inspections: {len(last):,}")
print(f"R², property's earlier mean score:                         {ols_r2(a):.3f}")
print(f"R², this visit's inspector + year effect:                  {ols_r2(ie + fe):.3f}")
print(f"R², both:                                                  {ols_r2(a, ie + fe):.3f}")
print(f"R², earlier inspector-adjusted mean + this visit's effect: {ols_r2(a_adj, ie + fe):.3f}")

In [ ]:
# Does bottom-10 status repeat? Consecutive inspections in different fiscal years.
s = ins_sorted.assign(prev_b10=ins_sorted.groupby("address").bottom10.shift(),
                      prev_fy=ins_sorted.groupby("address").fiscal_year.shift(),
                      prev_insp=ins_sorted.groupby("address").inspector.shift())
pairs = s.dropna(subset=["prev_b10"])
pairs = pairs[pairs.fiscal_year != pairs.prev_fy]
same = (pairs.inspector >= 0) & (pairs.inspector == pairs.prev_insp)
diff = (pairs.inspector >= 0) & (pairs.prev_insp >= 0) & (pairs.inspector != pairs.prev_insp)
tbl = pd.DataFrame({
    "pairs": [len(pairs), same.sum(), diff.sum()],
    "P(b10 | previous b10)": [pairs.loc[pairs.prev_b10 == 1, "bottom10"].mean(),
                              pairs.loc[same & (pairs.prev_b10 == 1), "bottom10"].mean(),
                              pairs.loc[diff & (pairs.prev_b10 == 1), "bottom10"].mean()],
    "P(b10 | previous not)": [pairs.loc[pairs.prev_b10 == 0, "bottom10"].mean(),
                              pairs.loc[same & (pairs.prev_b10 == 0), "bottom10"].mean(),
                              pairs.loc[diff & (pairs.prev_b10 == 0), "bottom10"].mean()],
}, index=["all consecutive pairs", "same inspector", "different inspector"])
print(tbl.round(3).to_string())

## 2. Inspector-adjusted target

Each score minus its inspector and fiscal-year effects (two-way fit on all inspections),
then "adjusted bottom 10%" = adjusted score at or below the 10th percentile of adjusted
scores. Property level: ever adjusted bottom 10%. This is a diagnostic. The effects use
every inspection, including test-area ones, so it describes how predictable inspector-free
condition is; it is not a deployable target.

In [ ]:
eff_i_all, eff_fy_all = two_way_effects(insp)
insp["adj_score"] = insp.score - insp.inspector.map(eff_i_all).fillna(0) - insp.fiscal_year.map(eff_fy_all).fillna(0)
insp["adj_b10"] = (insp.adj_score <= insp.adj_score.quantile(0.10)).astype(int)
prop_adj = insp.groupby("address").agg(adj_bottom10=("adj_b10", "max"), adj_min=("adj_score", "min"))

df = base.merge(prop_adj, left_on="address", right_index=True, how="left")
print("ever bottom-10, raw vs adjusted:", round(df.bottom10.mean(), 3), round(df.adj_bottom10.mean(), 3),
      "| agreement:", round((df.bottom10 == df.adj_bottom10).mean(), 3))

CONTEXT = f.spatial_context(df)
HIST = f.dcad_history(df)
df = pd.concat([df, CONTEXT, f.owner_keys(df)[["log_sponsor_portfolio"]], f.evictions(df), HIST,
                f.tax_delinquency(df), f.complaints_311(df)], axis=1)
PRE_ALL = (
    h.BASELINE_FEATURES + list(CONTEXT.columns) + ["log_sponsor_portfolio"]
    + ["ev_pre", "ev_pre_strong", "ev_pre_adj"] + [c for c in HIST.columns if c != "sold_during_window"]
    + ["delinquent_since_pre2021", "c311_pre", "c311_hazard_pre"]
)
FOLD = f.combine(f.owner_track_record, f.nearby_outcomes)
adj = pd.concat([
    h.run(df, "bottom10", PRE_ALL, fold_features=FOLD, label="raw target"),
    h.run(df, "adj_bottom10", PRE_ALL, fold_features=FOLD, label="inspector-adjusted target"),
], ignore_index=True)
adj["lift"] = adj["PR-AUC"] / adj.prevalence_test
print(adj[adj.ranker.isin(["Random", "Poverty only", "Ensemble"])].pivot_table(
    index=["label", "ranker"], columns="split", values=["PR-AUC", "lift"], sort=False).round(3).to_string())

## 3. Inspector leniency in the temporal design

In [ ]:
COSTAR, costar_owner = f.costar(base)
tbase = pd.concat([base, f.spatial_context(base), f.dcad_history(base), f.tax_delinquency(base), COSTAR], axis=1)
units = t.build_units(insp, tbase, owner_keys={"costar_owner": costar_owner})
units = t.add_inspector_leniency(units, insp)
units = units.merge(insp.groupby(["address", "fy"]).head(1)[["address", "fy", "adj_b10"]], on=["address", "fy"], how="left")
BEST = h.BASELINE_FEATURES + t.HISTORY_COLS   # notebook 05's best configuration
BASELINES = {"Last score (lower first)": -units.last_score.fillna(units.last_score.median()),
             "Inspector leniency only": units.inspector_prior_b10_rate}
temp = pd.concat([
    t.run(units, BEST, label="static + own history", baselines=BASELINES),
    t.run(units, BEST + t.INSPECTOR_COLS, label="+ inspector leniency"),
    t.run(units.assign(target=units.adj_b10), BEST, label="inspector-adjusted target"),
], ignore_index=True)
temp["lift"] = temp["PR-AUC"] / temp.prevalence_test
print(temp[temp.ranker.isin(["Random", "Last score (lower first)", "Inspector leniency only", "Ensemble"])].pivot_table(
    index=["label", "ranker"], columns="test", values=["PR-AUC", "lift"], sort=False).round(3).to_string())

## 4. Regression on the continuous score

In [ ]:
reg = pd.concat([
    h.run(df, "bottom10", PRE_ALL, fold_features=FOLD, label="classify bottom-10", which={"Chelsea mirror", "Grouped CV by tract"}),
    h.run(df, "bottom10", PRE_ALL, fold_features=FOLD, label="regress min score", models=h.REGRESSORS,
          regress_on="score_min", which={"Chelsea mirror", "Grouped CV by tract"}),
], ignore_index=True)
print(h.compare(reg[~reg.ranker.isin(["Random", "Poverty only"])]).round(3).to_string())
treg = pd.concat([
    t.run(units, BEST, label="classify bottom-10"),
    t.run(units, BEST, label="regress unit score", models=h.REGRESSORS, regress_on="unit_score"),
], ignore_index=True)
print(treg[treg.test == "Pooled"].pivot_table(index="label", columns="ranker", values="PR-AUC", sort=False).round(3).to_string())